# Adaptive RAG Experimental Notebook

This notebook documents the full experimental pipeline used in our project. The structure mirrors the final report, progressing from setup and method definition to evaluation and result analysis.


## Introduction and Experiment Overview

# Run everything: Dataset -> Corpus -> Indexing -> RAG → Adaptive RAG -> (optional) DPO → Eval

This notebook is designed for RunPod where your code lives in `/workspace`. 

Set `USE_OPENAI=True` if you want OpenAI for generation/rewriting/judging. Otherwise the notebook uses a local HF instruct model for generation and (optionally) DPO.


In [1]:
# Importing libraries and shared utilities
import sys
from pathlib import Path

# Find repo root by walking up until we see the "workspace" folder
p = Path.cwd().resolve()
while p != p.parent and not (p / "workspace").exists():
    p = p.parent

assert (p / "workspace").exists(), f"Couldn't find repo root above {Path.cwd().resolve()}"

REPO_ROOT = p
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Ensure package init files exist
(REPO_ROOT / "workspace" / "__init__.py").touch(exist_ok=True)
for sub in ["data", "retrieval", "models", "rlhf", "evaluation", "utils"]:
    (REPO_ROOT / "workspace" / sub / "__init__.py").touch(exist_ok=True)

print("✅ REPO_ROOT =", REPO_ROOT)
print("✅ sys.path[0] =", sys.path[0])


✅ REPO_ROOT = /workspace
✅ sys.path[0] = /workspace


In [ ]:
# Importing libraries and shared utilities (YOU MUST ADD AN API KEY)
import os
from pathlib import Path

os.environ["OPENAI_API_KEY"] = "YOUR-API-KEY HERE" 

ARTIFACTS = REPO_ROOT / "artifacts"
DATA_DIR = ARTIFACTS / "benchmarks"
CORPUS_DIR = ARTIFACTS / "corpus"
INDEX_DIR = ARTIFACTS / "indexes"
EVAL_DIR = ARTIFACTS / "eval_runs"
PREFS_DIR = ARTIFACTS / "prefs"
MODELS_DIR = ARTIFACTS / "models"

for p in [DATA_DIR, CORPUS_DIR, INDEX_DIR, EVAL_DIR, PREFS_DIR, MODELS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Artifacts dir:", ARTIFACTS)

HF_ID = "virattt/financial-qa-10K"

# Embeddings model
EMBED_MODEL = "BAAI/bge-small-en-v1.5"

# Local generator model (use for baseline + DPO)
LOCAL_GEN_MODEL = "Qwen/Qwen2.5-3B-Instruct"

# Optional OpenAI usage
USE_OPENAI = bool(os.getenv("OPENAI_API_KEY"))  # auto-enable if key exists
OPENAI_MODEL = "gpt-4o-mini"

print("USE_OPENAI =", USE_OPENAI)


Artifacts dir: /workspace/artifacts
USE_OPENAI = True


In [ ]:
import sys
from pathlib import Path

# Add repo root to PYTHONPATH
REPO_ROOT = Path.cwd().resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("Added to PYTHONPATH:", REPO_ROOT)


Added to PYTHONPATH: /workspace/notebooks


In [ ]:
# Installs All Dependencies
!pip install -r requirements.txt


[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip


In [ ]:
# Downloads dataset splits to parquet
from workspace.data.download_dataset import download_financial_qa_10k

download_financial_qa_10k(
    hf_id=HF_ID,
    out_dir=str(DATA_DIR),
    splits=("train","test"),
)

train_parquet = DATA_DIR / f"{HF_ID.replace('/','__')}_train.parquet"
test_parquet  = DATA_DIR / f"{HF_ID.replace('/','__')}_test.parquet"
train_parquet, test_parquet


2025-12-12 06:37:21,085 | data.download_dataset | INFO | Available splits for virattt/financial-qa-10K: ['train']
2025-12-12 06:37:21,086 | data.download_dataset | INFO | Downloading virattt/financial-qa-10K split=train
2025-12-12 06:37:21,640 | data.download_dataset | INFO | Saved train -> /workspace/artifacts/benchmarks/virattt__financial-qa-10K_train.parquet
2025-12-12 06:37:21,641 | data.download_dataset | WARNING | Split 'test' not found. Skipping.


(PosixPath('/workspace/artifacts/benchmarks/virattt__financial-qa-10K_train.parquet'),
 PosixPath('/workspace/artifacts/benchmarks/virattt__financial-qa-10K_test.parquet'))

In [ ]:
import pandas as pd

# If the dataset only has train, create a deterministic 80/20 split
if not test_parquet.exists():
    df = pd.read_parquet(train_parquet)

    # shuffle deterministically
    df = df.sample(frac=1.0, random_state=42).reset_index(drop=True)

    cut = int(0.8 * len(df))
    df_train = df.iloc[:cut].copy()
    df_test  = df.iloc[cut:].copy()

    # overwrite train_parquet with the 80% train split (optional but recommended)
    df_train.to_parquet(train_parquet, index=False)
    df_test.to_parquet(test_parquet, index=False)

    print("Created splits:")
    print("train:", len(df_train), "->", train_parquet)
    print("test :", len(df_test),  "->", test_parquet)
else:
    print("Test parquet already exists:", test_parquet)


Created splits:
train: 5600 -> /workspace/artifacts/benchmarks/virattt__financial-qa-10K_train.parquet
test : 1400 -> /workspace/artifacts/benchmarks/virattt__financial-qa-10K_test.parquet


In [9]:
# Training or fine-tuning models using the selected method
print("train_parquet =", train_parquet, train_parquet.exists())
print("test_parquet  =", test_parquet,  test_parquet.exists())


train_parquet = /workspace/artifacts/benchmarks/virattt__financial-qa-10K_train.parquet True
test_parquet  = /workspace/artifacts/benchmarks/virattt__financial-qa-10K_test.parquet True


In [ ]:
# Builds retrieval corpus (chunks) from both splits (more coverage)
from workspace.data.build_corpus import build_corpus_from_parquet

CHUNKS_PATH = CORPUS_DIR / "chunks.jsonl"

build_corpus_from_parquet(
    parquet_paths=[str(train_parquet), str(test_parquet)],
    out_chunks_path=str(CHUNKS_PATH),
    chunk_size_chars=1800,
    chunk_overlap_chars=200,
    min_chars=200,
)

CHUNKS_PATH


2025-12-12 06:39:54,468 | data.build_corpus | INFO | Unique contexts collected: 2881
2025-12-12 06:39:54,495 | data.build_corpus | INFO | Wrote chunks: 2888 -> /workspace/artifacts/corpus/chunks.jsonl


PosixPath('/workspace/artifacts/corpus/chunks.jsonl')

In [ ]:
# Build FAISS index
from workspace.retrieval.faiss_index import build_faiss_index

META_PATH = INDEX_DIR / "chunks_meta.jsonl"
FAISS_PATH = INDEX_DIR / "faiss_index.bin"

build_faiss_index(
    chunks_path=str(CHUNKS_PATH),
    meta_out_path=str(META_PATH),
    index_out_path=str(FAISS_PATH),
    embeddings_out_path=None,
    model_name=EMBED_MODEL,
    device="cuda",
    batch_size=64,
    normalize=True,
)

FAISS_PATH


2025-12-12 06:40:21,191 | retrieval.faiss_index | INFO | Loaded chunks: 2888


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

2025-12-12 06:40:25,171 | retrieval.faiss_index | INFO | Embedding chunks with BAAI/bge-small-en-v1.5


Batches:   0%|          | 0/46 [00:00<?, ?it/s]

2025-12-12 06:40:27,008 | retrieval.faiss_index | INFO | Embeddings shape: (2888, 384)
2025-12-12 06:40:27,074 | retrieval.faiss_index | INFO | Saved meta -> /workspace/artifacts/indexes/chunks_meta.jsonl
2025-12-12 06:40:27,075 | retrieval.faiss_index | INFO | Saved index -> /workspace/artifacts/indexes/faiss_index.bin


PosixPath('/workspace/artifacts/indexes/faiss_index.bin')

In [ ]:
# Builds evaluation benchmark JSONLs (train for DPO candidates, test for eval)
from workspace.data.build_benchmark import build_benchmark_from_parquet

TRAIN_JSONL = DATA_DIR / "train.jsonl"
TEST_JSONL = DATA_DIR / "test.jsonl"

build_benchmark_from_parquet(str(train_parquet), str(TRAIN_JSONL), max_examples=None)
build_benchmark_from_parquet(str(test_parquet), str(TEST_JSONL), max_examples=None)

TRAIN_JSONL, TEST_JSONL


2025-12-12 06:40:29,482 | data.build_benchmark | INFO | Saved benchmark 5600 rows -> /workspace/artifacts/benchmarks/train.jsonl
2025-12-12 06:40:29,567 | data.build_benchmark | INFO | Saved benchmark 1400 rows -> /workspace/artifacts/benchmarks/test.jsonl


(PosixPath('/workspace/artifacts/benchmarks/train.jsonl'),
 PosixPath('/workspace/artifacts/benchmarks/test.jsonl'))

In [ ]:
# Instantiates retrieval and corpus store
from workspace.retrieval.faiss_index import FaissSearcher
from workspace.retrieval.corpus_store import CorpusStore

searcher = FaissSearcher(
    index_path=str(FAISS_PATH),
    meta_path=str(META_PATH),
    embed_model=EMBED_MODEL,
    device="cuda",
)
store = CorpusStore(str(CHUNKS_PATH))


2025-12-12 06:40:33,118 | retrieval.corpus_store | INFO | Loaded chunk texts: 2888


In [ ]:
# Instantiates generator + rewriter clients (OpenAI or local)
from workspace.models.rag import RAGGenerator, QueryRewriter, AdaptiveRAG, RagConfig

if USE_OPENAI:
    from workspace.models.openai_client import OpenAIChat
    chat = OpenAIChat(model=OPENAI_MODEL)
else:
    from workspace.models.local_generator import LocalChat
    chat = LocalChat(model_name=LOCAL_GEN_MODEL)

generator = RAGGenerator(chat)
rewriter = QueryRewriter(chat)

cfg = RagConfig(
    top_k=20,
    context_k=5,
    confidence_threshold=0.5,
    max_adaptive_steps=3,
    alpha_gen=0.6,
)
rag = AdaptiveRAG(searcher=searcher, store=store, generator=generator, rewriter=rewriter, cfg=cfg)


In [ ]:
# Quick sanity check on a couple test questions
import json, itertools
from pprint import pprint

with open(TEST_JSONL, "r", encoding="utf-8") as f:
    exs = [json.loads(next(f)) for _ in range(3)]

for ex in exs:
    res_b = rag.answer_baseline(ex["question"])
    res_a = rag.answer_adaptive(ex["question"])
    print("\nQ:", ex["question"])
    print("Gold:", ex["answer"])
    print("Baseline:", res_b["answer"], "conf=", res_b["confidence"], "triggered=", res_b["adaptive_triggered"])
    print("Adaptive :", res_a["answer"], "conf=", res_a["confidence"], "triggered=", res_a["adaptive_triggered"])
    if res_a["adaptive_triggered"]:
        print("Rewritten:", res_a["rewritten_query"])


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Q: What criteria are used to classify loans and leases as nonperforming according to the described credit policy?
Gold: Loans and leases are classified as nonperforming when they are on nonaccrual status, such as being 90 days past due, have confirmed fraud or bankruptcy, or fit certain criteria such as being uninsured past a certain delinquency threshold or not well-secured and in the process of collection in the case of commercial loans.
Baseline: Loans and leases are classified as nonperforming if they are 90 days past due, have confirmed cases of fraud or bankruptcy, or specific types like consumer real estate-secured loans unless fully insured. Commercial loans and leases are also classified as nonperforming when past due 90 days or more unless well-secured and in the process of collection. conf= 0.9428465843200684 triggered= False
Adaptive : Loans and leases are classified as nonperforming if they are 90 days past due, have confirmed cases of fraud or bankruptcy, or specific typ

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2025-12-12 06:40:53,013 | models.rag | INFO | Adaptive: rewriting 'What are Etsy's main operating marketplaces as of 2023?' -> 'What are the main operating marketplaces of Etsy in 2023?'


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Q: What are Etsy's main operating marketplaces as of 2023?
Gold: As of 2023, Etsy's main operating marketplaces are Etsy.com, Reverb, and Depop.
Baseline: CANNOT_ANSWER conf= 0.32665786743164066 triggered= False
Adaptive : CANNOT_ANSWER conf= 0.32434766292572026 triggered= True
Rewritten: What are the main operating marketplaces of Etsy in 2023?


In [ ]:
# Evaluates baseline and adaptive with base generator
from workspace.evaluation.eval_runner import run_eval

summ_baseline = run_eval(
    benchmark_jsonl=str(TEST_JSONL),
    answer_fn=rag.answer_baseline,
    out_path=str(EVAL_DIR / "test_baseline_base.json"),
    max_examples=200,
)
summ_adaptive = run_eval(
    benchmark_jsonl=str(TEST_JSONL),
    answer_fn=rag.answer_adaptive,
    out_path=str(EVAL_DIR / "test_adaptive_base.json"),
    max_examples=200,
)

summ_baseline, summ_adaptive


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2025-12-12 06:51:19,526 | models.rag | INFO | Adaptive: rewriting 'What was the percentage decrease in sales for Alphagan/Combigan in the United States from 2021 to 2023?' -> 'What was the percentage decrease in sales for Alphagan/Combigan in the United States from 2021 to 2023?'


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2025-12-12 06:51:21,063 | evaluation.eval_runner | INFO | Eval: EM=0.035 F1=0.295 (n=200)
2025-12-12 06:51:21,072 | evaluation.eval_runner | INFO | Saved eval -> /workspace/artifacts/eval_runs/test_adaptive_base.json


({'n': 200, 'EM': 0.035, 'F1': 0.28323554158829434},
 {'n': 200, 'EM': 0.035, 'F1': 0.2947892987218169})

## Optional: DPO alignment (local generator only)

If you use OpenAI for generation, you can still use OpenAI as a **judge**, but DPO training produces a **local** LoRA adapter to apply to the local HF model.

If `USE_OPENAI=True`, you can still run DPO *provided you have a local model to train*. The code below uses `LOCAL_GEN_MODEL`.


In [19]:
# Training or fine-tuning models using the selected method
DO_DPO = True  # set False to skip


In [ ]:
# Generates candidate answer pairs from TRAIN set
if DO_DPO:
    from workspace.rlhf.gen_candidates import gen_candidates

    CANDS_PATH = PREFS_DIR / "candidates.jsonl"
    gen_candidates(
        dataset_jsonl=str(TRAIN_JSONL),
        searcher=searcher,
        store=store,
        chat_client=chat,
        out_path=str(CANDS_PATH),
        n=300,
        top_k=20,
        context_k=5,
    )
    CANDS_PATH


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2025-12-12 07:02:46,034 | rlhf.gen_candidates | INFO | Wrote 300 candidate pairs -> /workspace/artifacts/prefs/candidates.jsonl


In [ ]:
# Judge pairs to create DPO dataset
if DO_DPO:
    from workspace.rlhf.judge_pairs import judge_pairs

    if USE_OPENAI:
        # use OpenAI as judge
        judge_client = chat
    else:
        # local judge (weaker); still works but noisier
        judge_client = chat

    DPO_DATA = PREFS_DIR / "dpo_dataset.jsonl"
    judge_pairs(
        candidates_jsonl=str(CANDS_PATH),
        judge_client=judge_client,
        out_dpo_jsonl=str(DPO_DATA),
        max_pairs=200,
    )
    DPO_DATA


2025-12-12 07:10:13,544 | rlhf.judge_pairs | INFO | Judged 300 pairs, kept 49 -> /workspace/artifacts/prefs/dpo_dataset.jsonl


In [ ]:
# Trains DPO LoRA adapter
if DO_DPO:
    from workspace.rlhf.train_dpo_lora import train_dpo_lora

    DPO_OUT = MODELS_DIR / "dpo_lora_v1"
    train_dpo_lora(
        dpo_jsonl=str(DPO_DATA),
        base_model_name=LOCAL_GEN_MODEL,
        output_dir=str(DPO_OUT),
        max_rows=200,
        batch_size=2,
        grad_accum=8,
        lr=1e-5,
        epochs=1,
        beta=0.1,
    )
    DPO_OUT


2025-12-12 07:10:20,494 | rlhf.train_dpo_lora | INFO | Loaded DPO rows: 49


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Map:   0%|          | 0/49 [00:00<?, ? examples/s]

TypeError: DPOTrainer.__init__() got an unexpected keyword argument 'beta'

In [ ]:
# Importing libraries and shared utilities
if DO_DPO:
    # 12) Evaluate baseline/adaptive with DPO-aligned generator (local)
    from workspace.models.local_generator import LocalChat
    from workspace.models.rag import RAGGenerator, QueryRewriter, AdaptiveRAG, RagConfig

    chat_dpo = LocalChat(model_name=LOCAL_GEN_MODEL, lora_path=str(DPO_OUT))
    generator_dpo = RAGGenerator(chat_dpo)
    # rewriting can still use OpenAI or base local; keep same as before
    rewriter_same = QueryRewriter(chat)

    rag_dpo = AdaptiveRAG(
        searcher=searcher,
        store=store,
        generator=generator_dpo,
        rewriter=rewriter_same,
        cfg=cfg,
    )

    summ_baseline_dpo = run_eval(
        benchmark_jsonl=str(TEST_JSONL),
        answer_fn=rag_dpo.answer_baseline,
        out_path=str(EVAL_DIR / "test_baseline_dpo.json"),
        max_examples=200,
    )
    summ_adaptive_dpo = run_eval(
        benchmark_jsonl=str(TEST_JSONL),
        answer_fn=rag_dpo.answer_adaptive,
        out_path=str(EVAL_DIR / "test_adaptive_dpo.json"),
        max_examples=200,
    )

    summ_baseline_dpo, summ_adaptive_dpo


In [ ]:
# Displays a compact summary table
import pandas as pd

rows = []
rows.append({"variant":"baseline_base", **summ_baseline})
rows.append({"variant":"adaptive_base", **summ_adaptive})

pd.DataFrame(rows)


,variant,n,EM,F1
0,baseline_base,200,0.035,0.283236
1,adaptive_base,200,0.035,0.294789
